# PART 1 - Database Construction:
#### Objective: to convert vertical bunch of info. directly into a spreadsheet format
Instructions:
1. Replace .xlsx w/ new info.
2. Change the date
3. thats it.-

In [1]:
import pandas as pd
data_raw = pd.read_excel("ADL_Airport.xlsx", header=None)
data_raw.head()

,0
0,QF671
1,Melbourne
2,Est: 6:56am
3,Gate: 14
4,landed


In [3]:
data_list = data_raw[0].dropna().tolist()

In [5]:
#Checking data consistency:
def is_flight_number(item):
    flight_number = str(item).strip()
    return len(flight_number) > 0 and flight_number[0].isalpha() and flight_number[-1].isdigit() and " " not in flight_number

#Splitting into lists:
flights_raw = []
current = []

for item in data_list:
    if is_flight_number(item) and current:
        flights_raw.append(current)
        current = [item]
    else:
        current.append(item)
if current:
    flights_raw.append(current)

print(f"A total of {len(flights_raw)} flights were found in the spreadsheet.\n")

for f in flights_raw:
    if len(f) != 5:
        print(f"From those, showing unusual patterns are:\n - {len(f)} items: {f}")


A total of 91 flights were found in the spreadsheet.



In [7]:
#Fixing lists with more than 5 registers:

flights_list = []

for f in flights_raw:
    f = [x for x in f if not str(x).strip().lower().startswith("via")]
    flights_list.append(
        {"Flight": f[0].strip() if len(f) > 0 else "",
        "Destination": f[1].strip()                if len(f) > 1 else "",
        "Estimated Time": f[2].replace("Est:", "").strip() if len(f) > 2 else "",
        "Gate": f[3].replace("Gate:", "").strip()  if len(f) > 3 else "",
        "Status": f[4].strip()                     if len(f) > 4 else "unknown"})

data = pd.DataFrame(flights_list)
data.head()

,Flight,Destination,Estimated Time,Gate,Status
0,QF671,Melbourne,6:56am,14,landed
1,JQ776,Melbourne,8:05am,23,landed
2,VA211,Melbourne,7:55am,13,landed
3,JQ760,Sydney,8:02am,25,landed
4,VA404,Sydney,8:50am,14,landed


In [9]:
#Double checking data consistency:

print(f"Total flights built: {len(data)}")

unknown_status = (data["Status"] == "unknown").sum()
print(f"There were {unknown_status} filled 'status' column")

Total flights built: 91
There were 0 filled 'status' column


In [11]:
from datetime import date, timedelta
############# HERE ######################
fixed_date = date(2026,8,2) 
############# HERE ######################
data["Date"] = fixed_date.strftime("%d/%m/%Y")
data.head()

,Flight,Destination,Estimated Time,Gate,Status,Date
0,QF671,Melbourne,6:56am,14,landed,02/08/2026
1,JQ776,Melbourne,8:05am,23,landed,02/08/2026
2,VA211,Melbourne,7:55am,13,landed,02/08/2026
3,JQ760,Sydney,8:02am,25,landed,02/08/2026
4,VA404,Sydney,8:50am,14,landed,02/08/2026


In [13]:
consolidated_db = "ADL_aggregated_arrivals.csv"

import os

if os.path.exists(consolidated_db):
    exists = pd.read_csv(consolidated_db)
    if fixed_date.strftime("%d/%m/%Y") in exists["Date"].values:
        print(f"Already done, nothing was done (double check if this is not supposed to happen)")
    else:
        expected = len(flights_raw)
        data.to_csv(consolidated_db, mode = "a", header = False)
        print(f"Added {len(data)} rows (expected {expected}) as per {fixed_date.strftime('%d/%m/%Y')}")
else:
    data.to_csv(consolidated_db, mode = "w", header = True)
    print(f"Created FROM SCRATCH: {len(data)} as per {fixed_date.strftime('%d/%m/%Y')}")

Added 91 rows (expected 91) as per 02/08/2026


In [15]:
def categorise_aircraft(flight):
    letters = ''.join(k for k in str(flight) if k.isalpha()).upper()
    number  = ''.join(k for k in str(flight) if k.isdigit())
    if letters == "QF":
        if len(number) == 3:
            return "737"
        if number.startswith("19") or number.startswith("17") or number.startswith("18"):
            return "e190"
    if letters == "QQ":
        if len(number) == 4:
            return "e190"
    if letters == "VA":
        if len(number) == 3 or len(number) == 4:
            return "737"
    return "Insufficient information"
     

data["Aircraft"] = data["Flight"].apply(categorise_aircraft)

landed = data["Status"].astype(str).str.strip().str.lower().eq("landed")
gated  = data["Gate"].astype(str).str.strip().str.isdigit()
confirmed = data[landed & gated & data["Aircraft"].notna()].copy()
confirmed_perc = (len(confirmed)/len(data))*100
print(f"From {len(data)} total added flights, only {len(confirmed)} ({confirmed_perc:.2f} %) of them, can be categorised as:")
print(confirmed["Aircraft"].value_counts().to_string())

From 91 total added flights, only 77 (84.62 %) of them, can be categorised as:
Aircraft
737                         42
Insufficient information    24
e190                        11
